In [1]:
import subprocess, urllib.request, json, os

os.system('wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb')
os.system('apt-get install -y ./google-chrome-stable_current_amd64.deb -q 2>/dev/null')
os.system('pip install selenium selenium-stealth -q')

major = subprocess.check_output(['google-chrome','--version']).decode().split()[2].split('.')[0]
url = 'https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json'
with urllib.request.urlopen(url) as r:
    data = json.loads(r.read())

versions = [v for v in data['versions'] if v['version'].startswith(major + '.')]
latest = versions[-1]
driver_url = next(d['url'] for d in latest['downloads']['chromedriver'] if d['platform'] == 'linux64')
urllib.request.urlretrieve(driver_url, '/tmp/chromedriver.zip')
subprocess.run(['unzip', '-o', '/tmp/chromedriver.zip', '-d', '/tmp/cd'], capture_output=True)
result = subprocess.run(['find', '/tmp/cd', '-name', 'chromedriver', '-type', 'f'], capture_output=True, text=True)
subprocess.run(['cp', result.stdout.strip().split('\n')[0], '/usr/local/bin/chromedriver'])
subprocess.run(['chmod', '+x', '/usr/local/bin/chromedriver'])
print('Готово!')

Готово!


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium_stealth import stealth
import time

def create_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--ignore-certificate-errors')
    options.binary_location = '/usr/bin/google-chrome'
    service = Service(executable_path='/usr/local/bin/chromedriver')
    driver = webdriver.Chrome(service=service, options=options)
    stealth(driver,
        languages=['ru-RU', 'ru'],
        vendor='Google Inc.',
        platform='Win32',
        webgl_vendor='Intel Inc.',
        renderer='Intel Iris OpenGL Engine',
        fix_hairline=True,
    )
    return driver

driver = create_driver()
driver.get('https://www.avito.ru/moskva?q=огурцы+овощи')
time.sleep(7)
print(f'Title: {driver.title}')
body = driver.find_element(By.TAG_NAME, 'body').text
blocked = any(w in body[:300].lower() for w in
              ['403', 'forbidden', 'captcha', 'ограничен', 'vpn', 'бот'])
print(f'Статус: {"заблокирован" if blocked else "открылся!"}')
print(f'Первые 500 символов:\n{body[:500]}')
driver.quit()

Title: огурцы овощи - Купить продукты питания 🍱 в Москве | Недорогие продукты | Авито
Статус: открылся!
Первые 500 символов:
Как дальше пользоваться Авито на iOS.
Подробнее
Для бизнеса 
Карьера в Авито
Помощь
Ещё 
Избранное
Корзина
Вход и регистрация
Разместить объявление
Найти
Москва, район, метро
ГлавнаяДля дома и дачиПродукты питанияогурцы овощи
«Огурцы овощи»: объявления для Москвы65
 Сортировка 
Сначала из Москвы
Уведомлять о новых

Овощи, фрукты и Бакалея кафе, ресторан, ст
18 ₽
Москва
Новый Ассортимент. Пирожные макарони от 46р. ШТ. Доставка Бесплатно. Заказ от 5000р рублей. Доставка Москва и область 24/7. Дост


In [4]:
import re
import time

driver = create_driver()
driver.get('https://www.avito.ru/moskva?q=огурцы+овощи')
time.sleep(7)

print('Элементы с ценами')
all_els = driver.find_elements(By.XPATH, '//*[string-length(normalize-space(text())) > 0]')
found = 0
for el in all_els:
    txt = el.text.strip()
    cls = el.get_attribute('class') or ''
    if re.search(r'\b\d{2,5}\s*[₽р]', txt) and len(txt) < 50:
        print(f'  <{el.tag_name} class="{cls[:60]}">{txt}')
        found += 1
        if found > 20:
            break

driver.quit()
driver = create_driver()
driver.get('https://www.avito.ru/moskva?q=огурцы+овощи')
time.sleep(7)

# Смотрим структуру карточек
print('Элементы с ценами')
all_els = driver.find_elements(By.XPATH, '//*[string-length(normalize-space(text())) > 0]')
found = 0
for el in all_els:
    txt = el.text.strip()
    cls = el.get_attribute('class') or ''
    if re.search(r'\b\d{2,5}\s*[₽р]', txt) and len(txt) < 50:
        print(f'  <{el.tag_name} class="{cls[:60]}">{txt}')
        found += 1
        if found > 20:
            break

driver.quit()

Элементы с ценами
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">18 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">3 600 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">35 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">300 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">217 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">300 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">240 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">260 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">150 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">350 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">99 ₽
  <span class="styles-module-size_l-kPWfk styles-module-size_l_dense-kvYNM">550 ₽

In [9]:
import re, time, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium_stealth import stealth

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120


class AvitoParser:
    """
    Динамический парсинг цен на продукты с Авито (Москва).
    Список ингредиентов совпадает с данными Росстата —
    для корректного сравнения двух источников.
    """

    BASE_URL = 'https://www.avito.ru/moskva?q={query}'
    PRICE_CLASS = 'styles-module-size_l-kPWfk'

    # Те же 5 позиций что и в Росстате
    INGREDIENTS = [
        {'name': 'Куриное филе',      'query': 'филе+куриное+мясо',  'grams': 130},
        {'name': 'Яйца куриные',      'query': 'яйца+куриные',       'grams': 55},
        {'name': 'Огурцы свежие',     'query': 'огурцы+свежие',      'grams': 65},
        {'name': 'Помидоры свежие',   'query': 'помидоры+свежие',    'grams': 90},
        {'name': 'Масло подсолнечное','query': 'масло+подсолнечное', 'grams': 10},
    ]

    def __init__(self):
        self.driver  = None
        self.results = []
        self.df      = None

    def _create_driver(self):
        options = Options()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--ignore-certificate-errors')
        options.binary_location = '/usr/bin/google-chrome'
        service = Service(executable_path='/usr/local/bin/chromedriver')
        driver = webdriver.Chrome(service=service, options=options)
        stealth(driver,
            languages=['ru-RU', 'ru'],
            vendor='Google Inc.',
            platform='Win32',
            webgl_vendor='Intel Inc.',
            renderer='Intel Iris OpenGL Engine',
            fix_hairline=True,
        )
        return driver

    def start(self):
        self.driver = self._create_driver()
        print('Браузер запущен')
        return self

    def stop(self):
        if self.driver:
            self.driver.quit()
            print('Браузер закрыт')
        return self

    def _parse_prices(self, query):
        url = self.BASE_URL.format(query=query)
        self.driver.get(url)
        time.sleep(random.uniform(5, 7))

        price_elements = self.driver.find_elements(
            By.CSS_SELECTOR, f'[class*="{self.PRICE_CLASS}"]'
        )
        prices = []
        for el in price_elements:
            txt = el.text.strip().replace(' ', '')
            nums = re.findall(r'\d+', txt)
            if nums:
                price = int(nums[0])
                if 10 <= price <= 5000:
                    prices.append(price)
        return prices

    def parse_all(self):
        print('\n Парсим Авито (Москва)')
        print('─' * 80)
        for ing in self.INGREDIENTS:
            print(f'{ing["name"]:<25}', end=' ', flush=True)
            prices = self._parse_prices(ing['query'])

            if prices:
                median = float(np.median(prices))
                self.results.append({
                    'Ингредиент':          ing['name'],
                    'Граммов в тарелке':   ing['grams'],
                    'Мин. цена (₽/кг)':    float(min(prices)),
                    'Медиана (₽/кг)':      median,
                    'Макс. цена (₽/кг)':   float(max(prices)),
                    'Цена ₽/100г':         round(median / 10, 2),
                    'Стоимость в тарелке': round(median / 10 * ing['grams'] / 100, 2),
                    'Кол-во объявлений':   len(prices),
                })
                print(f' медиана {median:.0f} ₽/кг  '
                      f'(мин {min(prices)} — макс {max(prices)})  '
                      f'[{len(prices)} объявлений]')
            else:
                print('не найдено')

            time.sleep(random.uniform(3, 5))

        self.df = pd.DataFrame(self.results)
        self.df.to_csv('/content/avito_prices.csv', index=False)
        print('Данные Авито сохранены в avito_prices.csv')
        return self

    def print_table(self):
        total   = self.df['Стоимость в тарелке'].sum()
        total_g = self.df['Граммов в тарелке'].sum()

        print(f'\n Результаты парсинга Авито (Москва):')
        print('─' * 70)
        print(f'{"Ингредиент":<22} {"г":>4} {"Мин":>6} '
              f'{"Медиана":>8} {"Макс":>6} {"₽/100г":>8} {"В тарелке":>10}')
        print('─' * 70)
        for _, row in self.df.iterrows():
            print(f'{row["Ингредиент"]:<22} '
                  f'{row["Граммов в тарелке"]:>4} '
                  f'{row["Мин. цена (₽/кг)"]:>6.0f} '
                  f'{row["Медиана (₽/кг)"]:>8.0f} '
                  f'{row["Макс. цена (₽/кг)"]:>6.0f} '
                  f'{row["Цена ₽/100г"]:>8.2f} '
                  f'{row["Стоимость в тарелке"]:>10.2f}')
        print('─' * 70)
        print(f'{"ИТОГО":<22} {total_g:>4} {"":>28} {total:>10.2f} ₽')
        print(f'\n Себестоимость {total_g}г (Авито): {total:.2f} ₽')
        return self

